# Last-Layer Embedding Substrate Annotation Workflow

This notebook is a Colab-friendly front end for the last-layer ESM3 embedding training workflow.

The goal is to test whether ESM3 final hidden-layer representations can be used as features for predicting substrate-related enzyme labels.

Instead of using ESM3 `function_logits`, this workflow uses per-residue last-layer embeddings saved as tensors with shape:

```text
[residues, hidden_dim]
[436, 1536]


In [ ]:
# Optional: mount Google Drive in Colab
# Uncomment if your ML folder is stored in Drive.

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. Setup

Set the path to the folder that contains the scripts and datasets.

In Colab, this is often a Google Drive path.

In [ ]:
from pathlib import Path
import os
import json
import subprocess
from datetime import datetime
from IPython.display import Image
import sys
import pandas as pd
import random
import numpy as np
import importlib
import torch
import datetime as dt
import importlib.util
import sys
import subprocess
import sys, importlib
import classifier_screen
from scipy.stats import loguniform

# Example:
# ML_DIR = Path("/content/drive/MyDrive/your_project/code_and_data")

# Add CS_5782_Final_Project as a shortcut
# to your /MyDrive folder so we can all
# use the same path below.

ML_DIR = Path("/content/drive/MyDrive/CS_5782_Final_Project/code_and_data")

assert ML_DIR.exists(), f"ML_DIR does not exist: {ML_DIR}"
os.chdir(ML_DIR)
print("Working directory:", ML_DIR.resolve())

Working directory: /content/drive/MyDrive/CS_5782_Final_Project/code_and_data


In [ ]:
#DEFINE RUN HELPER FUNCITON
def run_cmd(cmd):
    print("\n$", " ".join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=ML_DIR,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")

In [ ]:
#CHECK IF DATA IS PRESENT
print("Files present:")

expected_items = [
    "utils.py",
    "train_lr_embedding_baseline.py",
    "train_mlp_embedding_baseline.py",
    "train_embedding_transformer.py",
    "BAHD_dataset",
    "UGT_dataset",
]

for name in expected_items:
    path = ML_DIR / name
    print(f"  {name}:", "OK" if path.exists() else "MISSING")


Files present:
  utils.py: OK
  train_lr_embedding_baseline.py: OK
  train_mlp_embedding_baseline.py: OK
  train_embedding_transformer.py: OK
  BAHD_dataset: OK
  UGT_dataset: OK


In [ ]:
#VERIFY PACKAGES
required = ["numpy", "torch", "sklearn", "pandas"]

for pkg in required:
    importlib.import_module(pkg)
    print(f"{pkg}: OK")

numpy: OK
torch: OK
sklearn: OK
pandas: OK


In [ ]:
#CHECK CUDA & GPU

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device 0:", torch.cuda.get_device_name(0))


CUDA available: False


## 2. Common Run Settings

Change these values once, then reuse them for all runs.

In [ ]:
DATASET = "BAHD"          # BAHD, UGT, or both
CV_MODE = "kfold"         # kfold or loocv
N_SPLITS = 5              # used when CV_MODE == "kfold"
DEVICE = "auto"           # auto, cpu, cuda, cuda:0
MIN_POSITIVE_COUNT = 2
WRITE_PREDICTIONS = True
N_RUNS = 15   # random search runs per model
RS_SEED = 42  # seed for parameter sampling reproducibility
SAVE_MODELS = True
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")              #timestamp tag
LAST_LAYER_STEPS = 10                                           #10 steps for this dataset
LAST_LAYER_GLOB = f"*_hidden_layer_steps{LAST_LAYER_STEPS}.pt"  #for finding run


#Dataset Folders
BAHD_DATASET_DIR = ML_DIR / "BAHD_dataset"
UGT_DATASET_DIR = ML_DIR / "UGT_dataset"
BAHD_EMBEDDING_DIR = ML_DIR / "BAHD_lastLayer_embeddings"
UGT_EMBEDDING_DIR = ML_DIR / "UGT_lastLayer_embeddings"

#Output Folders
LR_OUTPUT_DIR = ML_DIR / "results" / "lr_embedding_baseline"
MLP_OUTPUT_DIR = ML_DIR / "results" / "mlp_embedding_baseline"
TRANSFORMER_OUTPUT_DIR = ML_DIR / "results" / "embedding_transformer"

settings = {
    "dataset": DATASET,
    "cv_mode": CV_MODE,
    "n_splits": N_SPLITS,
    "device": DEVICE,
    "min_positive_count": MIN_POSITIVE_COUNT,
    "write_predictions": WRITE_PREDICTIONS,
    "save_models": SAVE_MODELS,
    "run_tag": RUN_TAG,
    "last_layer_steps": LAST_LAYER_STEPS,
    "last_layer_glob": LAST_LAYER_GLOB,
    "bahd_embedding_dir": str(BAHD_EMBEDDING_DIR),
    "ugt_embedding_dir": str(UGT_EMBEDDING_DIR),
    "lr_output_dir": str(LR_OUTPUT_DIR),
    "mlp_output_dir": str(MLP_OUTPUT_DIR),
    "transformer_output_dir": str(TRANSFORMER_OUTPUT_DIR),
}

print(json.dumps(settings, indent=2))

{
  "dataset": "BAHD",
  "cv_mode": "kfold",
  "n_splits": 5,
  "device": "auto",
  "min_positive_count": 2,
  "write_predictions": true,
  "save_models": true,
  "run_tag": "20260509_200234",
  "last_layer_steps": 10,
  "last_layer_glob": "*_hidden_layer_steps10.pt",
  "bahd_embedding_dir": "/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings",
  "ugt_embedding_dir": "/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings",
  "lr_output_dir": "/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/lr_embedding_baseline",
  "mlp_output_dir": "/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/mlp_embedding_baseline",
  "transformer_output_dir": "/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/embedding_transformer"
}


In [ ]:
#Helper Function

def make_run_tag(model_name: str, run_index: int) -> str:
    ts = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{model_name}_run{run_index:02d}_{ts}"

## 3. Exploratory - Logistic Regression Baseline (LL to Label)

In [ ]:

#NOTES
#   mean pools [510,1536] -> [1536]
#   standard scalar
#   OneVsRestClassifier LogisticRegression

run_cmd([
    "python", "train_lr_embedding_baseline.py",
    "--dataset", DATASET,
    "--bahd_dataset_dir", str(BAHD_DATASET_DIR),
    "--ugt_dataset_dir", str(UGT_DATASET_DIR),
    "--bahd_embedding_dir", str(BAHD_EMBEDDING_DIR),
    "--ugt_embedding_dir", str(UGT_EMBEDDING_DIR),
    "--embedding_glob", LAST_LAYER_GLOB,
    "--cv_mode", CV_MODE,
    "--n_splits", str(N_SPLITS),
    "--device", DEVICE,
    "--min_positive_count", str(MIN_POSITIVE_COUNT),
    "--output_dir", str(LR_OUTPUT_DIR),
    "--run_name", f"lr_lastlayer_{RUN_TAG}",
    "--write_predictions" if WRITE_PREDICTIONS else "--no-write_predictions",
    "--pickle_models" if SAVE_MODELS else "--no-pickle_models",
])


$ python train_lr_embedding_baseline.py --dataset BAHD --bahd_dataset_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_dataset --ugt_dataset_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_dataset --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --output_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/lr_embedding_baseline --run_name lr_lastlayer_20260427_210051 --write_predictions --pickle_models


## 4. Exploratory - MLP Baseline (LL to Label)

In [ ]:
MLP_POOLING = "mean"      # mean or attention
MLP_HIDDEN_DIM = 256
MLP_DROPOUT = 0.2
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_EPOCHS = 100
MLP_BATCH_SIZE = 16


In [ ]:
run_cmd([
    "python", "train_mlp_embedding_baseline.py",
    "--dataset", DATASET,
    "--bahd_dataset_dir", str(BAHD_DATASET_DIR),
    "--ugt_dataset_dir", str(UGT_DATASET_DIR),
    "--bahd_embedding_dir", str(BAHD_EMBEDDING_DIR),
    "--ugt_embedding_dir", str(UGT_EMBEDDING_DIR),
    "--embedding_glob", LAST_LAYER_GLOB,
    "--cv_mode", CV_MODE,
    "--n_splits", str(N_SPLITS),
    "--device", DEVICE,
    "--min_positive_count", str(MIN_POSITIVE_COUNT),
    "--pooling", MLP_POOLING,
    "--hidden_dim", str(MLP_HIDDEN_DIM),
    "--dropout", str(MLP_DROPOUT),
    "--learning_rate", str(MLP_LR),
    "--weight_decay", str(MLP_WEIGHT_DECAY),
    "--epochs", str(MLP_EPOCHS),
    "--batch_size", str(MLP_BATCH_SIZE),
    "--output_dir", str(MLP_OUTPUT_DIR),
    "--run_name", f"mlp_lastlayer_{MLP_POOLING}_{RUN_TAG}",
    "--write_predictions" if WRITE_PREDICTIONS else "--no-write_predictions",
    "--save_models" if SAVE_MODELS else "--no-save_models",
])


## 5. Exploratory - Last-Layer Transformer (LL to Label)

In [ ]:
# [residues, 1536]
# → linear projection to d_model
# → positional encoding
# → transformer encoder
# → mean or attention pooling
# → multilabel substrate prediction

TRANSFORMER_POOLING = "attention"   # mean or attention
TRANSFORMER_D_MODEL = 256
TRANSFORMER_N_HEADS = 4
TRANSFORMER_N_LAYERS = 2
TRANSFORMER_DROPOUT = 0.1
TRANSFORMER_LR = 1e-4
TRANSFORMER_WEIGHT_DECAY = 1e-4
TRANSFORMER_EPOCHS = 60
TRANSFORMER_BATCH_SIZE = 8


In [ ]:
run_cmd([
    "python", "train_embedding_transformer.py",
    "--dataset", DATASET,
    "--bahd_dataset_dir", str(BAHD_DATASET_DIR),
    "--ugt_dataset_dir", str(UGT_DATASET_DIR),
    "--bahd_embedding_dir", str(BAHD_EMBEDDING_DIR),
    "--ugt_embedding_dir", str(UGT_EMBEDDING_DIR),
    "--embedding_glob", LAST_LAYER_GLOB,
    "--cv_mode", CV_MODE,
    "--n_splits", str(N_SPLITS),
    "--device", DEVICE,
    "--min_positive_count", str(MIN_POSITIVE_COUNT),
    "--pooling", TRANSFORMER_POOLING,
    "--d_model", str(TRANSFORMER_D_MODEL),
    "--n_heads", str(TRANSFORMER_N_HEADS),
    "--n_layers", str(TRANSFORMER_N_LAYERS),
    "--dropout", str(TRANSFORMER_DROPOUT),
    "--learning_rate", str(TRANSFORMER_LR),
    "--weight_decay", str(TRANSFORMER_WEIGHT_DECAY),
    "--epochs", str(TRANSFORMER_EPOCHS),
    "--batch_size", str(TRANSFORMER_BATCH_SIZE),
    "--output_dir", str(TRANSFORMER_OUTPUT_DIR),
    "--run_name", f"embedding_transformer_{TRANSFORMER_POOLING}_{RUN_TAG}",
    "--write_predictions" if WRITE_PREDICTIONS else "--no-write_predictions",
    "--save_models" if SAVE_MODELS else "--no-save_models",
])


##6. Exploratory - LGBMClassifier (old)

In [ ]:
#INSTALL LGBM IF NEEDED
if importlib.util.find_spec("lightgbm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])

In [ ]:
LGBM_N_ESTIMATORS = 200
LGBM_LR = 0.05
LGBM_NUM_LEAVES = 31
LGBM_MAX_DEPTH = -1
LGBM_MIN_CHILD_SAMPLES = 10
LGBM_SUBSAMPLE = 0.8
LGBM_COLSAMPLE_BYTREE = 0.8
LGBM_REG_ALPHA = 0.0
LGBM_REG_LAMBDA = 1.0

In [ ]:
#This produces a ton of warnings but apparently according to ChatGPT it's not an issue
run_cmd([
    "python", "train_lgbm_embedding_baseline.py",
    "--dataset", DATASET,
    "--bahd_dataset_dir", str(BAHD_DATASET_DIR),
    "--ugt_dataset_dir", str(UGT_DATASET_DIR),
    "--bahd_embedding_dir", str(BAHD_EMBEDDING_DIR),
    "--ugt_embedding_dir", str(UGT_EMBEDDING_DIR),
    "--embedding_glob", LAST_LAYER_GLOB,
    "--cv_mode", CV_MODE,
    "--n_splits", str(N_SPLITS),
    "--device", DEVICE,
    "--min_positive_count", str(MIN_POSITIVE_COUNT),
    "--n_estimators", str(LGBM_N_ESTIMATORS),
    "--learning_rate", str(LGBM_LR),
    "--num_leaves", str(LGBM_NUM_LEAVES),
    "--max_depth", str(LGBM_MAX_DEPTH),
    "--min_child_samples", str(LGBM_MIN_CHILD_SAMPLES),
    "--subsample", str(LGBM_SUBSAMPLE),
    "--colsample_bytree", str(LGBM_COLSAMPLE_BYTREE),
    "--reg_alpha", str(LGBM_REG_ALPHA),
    "--reg_lambda", str(LGBM_REG_LAMBDA),
    "--output_dir", str(ML_DIR / "results" / "lgbm_embedding_baseline"),
    "--run_name", f"lgbm_lastlayer_{RUN_TAG}",
    "--write_predictions" if WRITE_PREDICTIONS else "--no-write_predictions",
    "--pickle_models" if SAVE_MODELS else "--no-pickle_models",
])


$ python train_lgbm_embedding_baseline.py --dataset BAHD --bahd_dataset_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_dataset --ugt_dataset_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_dataset --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --n_estimators 200 --learning_rate 0.05 --num_leaves 31 --max_depth -1 --min_child_samples 10 --subsample 0.8 --colsample_bytree 0.8 --reg_alpha 0.0 --reg_lambda 1.0 --output_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/lgbm_embedding_baseline --run_name lgbm_lastlayer_20260427_210612 --write_predictions --pickle_models
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does n

##7. Random Search - Logistic Regression

In [ ]:
rng    = random.Random(RS_SEED)
np_rng = np.random.default_rng(RS_SEED)
ll_lr_tags = []

for i in range(N_RUNS):
    C       = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(100)))
    penalty = rng.choice(['l1', 'l2'])
    tag     = make_run_tag('lr_embedding_baseline', i)
    ll_lr_tags.append(tag)

    print(f'\n--- LL-LR run {i:02d} | C={C:.5f} penalty={penalty} ---')
    run_cmd([
        'python', 'train_lr_embedding_baseline.py',
        '--dataset',            DATASET,
        '--bahd_embedding_dir', str(BAHD_EMBEDDING_DIR),
        '--ugt_embedding_dir',  str(UGT_EMBEDDING_DIR),
        '--embedding_glob',     LAST_LAYER_GLOB,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--C',                  f'{C:.8f}',
        '--penalty',            penalty,
        '--run_name',           tag,
        '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
        '--no-pickle_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nLL-LR random search complete.')


--- LL-LR run 00 | C=4.40287 penalty=l1 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --C 4.40287435 --penalty l1 --run_name lr_embedding_baseline_run00_20260428_231845 --write_predictions --no-pickle_models
  Run 00 done | donor micro_aupr = 0.9517
  Run 00 done | acceptor micro_aupr = 0.6275

--- LL-LR run 01 | C=0.04298 penalty=l1 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold -

##8. Random Search - Multi Layer Perceptron

In [ ]:
rng    = random.Random(RS_SEED)
np_rng = np.random.default_rng(RS_SEED)
ll_mlp_tags = []

#Variables to random search
VALID_HIDDEN_DIMS   = [64, 128, 256, 512]
VALID_DROPOUTS      = [0.0, 0.1, 0.2, 0.3, 0.5]
VALID_WEIGHT_DECAYS = [0, 1e-5, 1e-4, 1e-3]
VALID_POOLINGS      = ['mean', 'attention']

for i in range(N_RUNS):
    hidden_dim   = rng.choice(VALID_HIDDEN_DIMS)
    dropout      = rng.choice(VALID_DROPOUTS)
    lr           = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(1e-2)))
    weight_decay = rng.choice(VALID_WEIGHT_DECAYS)
    pooling      = rng.choice(VALID_POOLINGS)
    tag          = make_run_tag('mlp_embedding_baseline', i)
    ll_mlp_tags.append(tag)

    print(f'\n--- LL-MLP run {i:02d} | hidden={hidden_dim} drop={dropout} lr={lr:.5f} wd={weight_decay} pool={pooling} ---')
    run_cmd([
        'python', 'train_mlp_embedding_baseline.py',
        '--dataset',                 DATASET,
        '--bahd_embedding_dir',      str(BAHD_EMBEDDING_DIR),
        '--ugt_embedding_dir',       str(UGT_EMBEDDING_DIR),
        '--embedding_glob',          LAST_LAYER_GLOB,
        '--cv_mode',                 CV_MODE,
        '--n_splits',                str(N_SPLITS),
        '--device',                  DEVICE,
        '--min_positive_count',      str(MIN_POSITIVE_COUNT),
        '--hidden_dim',              str(hidden_dim),
        '--dropout',                 str(dropout),
        '--learning_rate',           f'{lr:.8f}',
        '--weight_decay',            str(weight_decay),
        '--pooling',                 pooling,
        '--epochs',                  '500',
        '--batch_size',              '16',
        '--early_stopping_patience', '10',
        '--run_name',                tag,
        '--write_predictions' if WRITE_PREDICTIONS else '--no-write_predictions',
        '--no-save_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'mlp_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nLL-MLP random search complete.')


--- LL-MLP run 00 | hidden=64 drop=0.0 lr=0.00353 wd=0.0001 pool=mean ---

$ python train_mlp_embedding_baseline.py --dataset BAHD --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --hidden_dim 64 --dropout 0.0 --learning_rate 0.00353112 --weight_decay 0.0001 --pooling mean --epochs 500 --batch_size 16 --early_stopping_patience 10 --run_name mlp_embedding_baseline_run00_20260429_001722 --write_predictions --no-save_models
  Run 00 done | donor micro_aupr = 0.9577
  Run 00 done | acceptor micro_aupr = 0.4806

--- LL-MLP run 01 | hidden=128 drop=0.1 lr=0.00075 wd=0 pool=mean ---

$ python train_mlp_embedding_baseline.py --dataset BAHD --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_last

##9. Random Search - Transformer

In [ ]:
#Variables to random search
VALID_D_NHEAD_PAIRS = [(128,4),(128,8),(256,4),(256,8),(512,4),(512,8)]
VALID_NUM_LAYERS    = [1, 2, 3]
VALID_DROPOUTS      = [0.0, 0.1, 0.2, 0.3]
VALID_POOLINGS      = ["mean", "attention"]
ll_tfm_tags = []

#Because the RNG is set (and collab keeps disconnecting) here is a way to only
#run certain indexes of the RNG seed

np_rng = np.random.default_rng(RS_SEED)

START_RUN = 11   # first run index to actually execute (default 0)
END_RUN   = 14   # last run index to actually execute  (default 14)

np_rng = np.random.default_rng(RS_SEED)

for i in range(max(N_RUNS, END_RUN + 1)):
    d_model, n_heads = VALID_D_NHEAD_PAIRS[np_rng.integers(len(VALID_D_NHEAD_PAIRS))]
    num_layers = int(VALID_NUM_LAYERS[np_rng.integers(len(VALID_NUM_LAYERS))])
    dropout    = float(VALID_DROPOUTS[np_rng.integers(len(VALID_DROPOUTS))])
    pooling    = VALID_POOLINGS[np_rng.integers(len(VALID_POOLINGS))]
    lr         = float(10 ** np_rng.uniform(np.log10(1e-4), np.log10(5e-3)))

    if i < START_RUN or i > END_RUN:
        print(f"[run {i:02d}] skipping")
        continue

    tag = make_run_tag("embedding_transformer", i)
    ll_tfm_tags.append(tag)

    print(f'\n--- LL-TFM run {i:02d} | d_model={d_model} n_heads={n_heads} '
          f'layers={num_layers} drop={dropout} pool={pooling} lr={lr:.5f} ---')
    run_cmd([
        'python', 'train_embedding_transformer.py',
        '--dataset',            DATASET,
        '--bahd_embedding_dir', str(BAHD_EMBEDDING_DIR),
        '--ugt_embedding_dir',  str(UGT_EMBEDDING_DIR),
        '--embedding_glob',     LAST_LAYER_GLOB,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--d_model',            str(d_model),
        '--n_heads',            str(n_heads),
        '--n_layers',           str(num_layers),
        '--dropout',            str(dropout),
        '--pooling',            pooling,
        '--learning_rate',      f'{lr:.8f}',
        '--epochs',             '100',
        '--batch_size',         '8',
        '--run_name',           tag,
        '--no-save_models',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'embedding_transformer' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done')

print('\nLL-TFM resume complete.')

[run 00] skipping
[run 01] skipping
[run 02] skipping
[run 03] skipping
[run 04] skipping
[run 05] skipping
[run 06] skipping
[run 07] skipping
[run 08] skipping
[run 09] skipping
[run 10] skipping

--- LL-TFM run 11 | d_model=512 n_heads=8 layers=2 drop=0.0 pool=mean lr=0.00021 ---

$ python train_embedding_transformer.py --dataset BAHD --bahd_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/BAHD_lastLayer_embeddings --ugt_embedding_dir /content/drive/MyDrive/CS_5782_Final_Project/code_and_data/UGT_lastLayer_embeddings --embedding_glob *_hidden_layer_steps10.pt --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --d_model 512 --n_heads 8 --n_layers 2 --dropout 0.0 --pooling mean --learning_rate 0.00020985 --epochs 100 --batch_size 8 --run_name embedding_transformer_run11_20260429_210417 --no-save_models
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage 

##10. Homebrewed LazyPredict Screening

In [ ]:

sys.path.insert(0, str(ML_DIR))


sys.argv = [
    "classifier_screen.py",
    "--feature_type", "last_layer",
    "--dataset", DATASET,
    "--bahd_dataset_dir", str(BAHD_DATASET_DIR),
    "--bahd_embedding_dir", str(BAHD_EMBEDDING_DIR),
    "--embedding_glob", LAST_LAYER_GLOB,
    "--cv_mode", CV_MODE,
    "--n_splits", str(N_SPLITS),
    "--random_seed", "42",
]

importlib.reload(classifier_screen)
classifier_screen.main()


  BAHD | donor | last_layer
  X shape      : (366, 1536)
  Labels       : 2
  CV splits    : 5

  [LogisticRegression] micro_aupr=0.9599
  [RidgeClassifier] micro_aupr=0.9434
  [LinearSVC] 

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

micro_aupr=0.9522
  [SGDClassifier] 

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


micro_aupr=0.9512
  [RandomForest] micro_aupr=0.9553
  [ExtraTrees] micro_aupr=0.9486
  [DecisionTree] micro_aupr=0.8167
  [HistGradientBoosting] micro_aupr=0.9596
  [KNeighbors] micro_aupr=0.9182
  [GaussianNB] micro_aupr=0.8345
  [BernoulliNB] micro_aupr=0.8290
  [SVC_RBF] micro_aupr=0.9538
  [LinearDiscriminantAnalysis] micro_aupr=0.8932
  [LightGBM] 

...

micro_aupr=0.6350
  [RandomForest] micro_aupr=0.6049
  [ExtraTrees] micro_aupr=0.5960
  [DecisionTree] micro_aupr=0.2689
  [HistGradientBoosting] micro_aupr=0.6404
  [KNeighbors] micro_aupr=0.5893
  [GaussianNB] micro_aupr=0.3219
  [BernoulliNB] micro_aupr=0.1872
  [SVC_RBF] micro_aupr=0.6681
  [LinearDiscriminantAnalysis] micro_aupr=0.5546
  [LightGBM] 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

micro_aupr=0.6279
  [XGBoost] 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

micro_aupr=0.6176

  Rank Classifier                     Micro AUPR Macro AUPR Micro AUROC
  -------------------------------------------------------------------
  1    SVC_RBF                            0.6681     0.5035      0.9028
  2    LinearSVC                          0.6677     0.5352      0.8848
  3    LogisticRegression                 0.6511     0.5195      0.9164
  4    HistGradientBoosting               0.6404     0.4987      0.9247
  5    SGDClassifier                      0.6350     0.4999      0.9002
  6    LightGBM                           0.6279     0.4924      0.9156
  7    RidgeClassifier                    0.6237     0.5376      0.8852
  8    XGBoost                            0.6176     0.4465      0.8864
  9    RandomForest                       0.6049     0.4458      0.9098
  10   ExtraTrees                         0.5960     0.4942      0.8986
  11   KNeighbors                         0.5893     0.4134      0.8759
  12   LinearDiscriminantAnalysis         0.554

## 11. Random Search - SVC_RBF (LazyPredict Winner)

In [ ]:


N_RUNS  = 15
RS_SEED = 42
np_rng  = np.random.RandomState(RS_SEED)
svc_ll_tags = []

for i in range(N_RUNS):
    C     = float(loguniform(1e-2, 1e3).rvs(random_state=np_rng))
    gamma = float(loguniform(1e-4, 1e0).rvs(random_state=np_rng))
    tag   = make_run_tag('svc_embedding_baseline', i)
    svc_ll_tags.append(tag)

    print(f'\n--- SVC (LL) run {i:02d} | C={C:.5f}  gamma={gamma:.6f} ---')
    run_cmd([
        'python', 'train_svc_embedding_baseline.py',
        '--dataset',            DATASET,
        '--C',                  f'{C:.8f}',
        '--gamma',              f'{gamma:.8f}',
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED),
        '--run_name',           tag,
        '--no-pickle_models',
        '--no-write_predictions',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'svc_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done (no metrics found)')

print('\nSVC (LL) random search complete.')
print(svc_ll_tags)


--- SVC (LL) run 00 | C=0.74593  gamma=0.635122 ---

$ python train_svc_embedding_baseline.py --dataset BAHD --C 0.74593433 --gamma 0.63512210 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 42 --run_name svc_embedding_baseline_run00_20260430_231951 --no-pickle_models --no-write_predictions

[BAHD | donor]  C=0.745934  gamma=0.6351221
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done

[BAHD | acceptor]  C=0.745934  gamma=0.6351221
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done

  Run 00 done | donor micro_aupr = 0.6895
  Run 00 done | acceptor micro_aupr = 0.2755

--- SVC (LL) run 01 | C=45.70563  gamma=0.024810 ---

$ python train_svc_embedding_baseline.py --dataset BAHD --C 45.70563100 --gamma 0.02481041 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 42 --run_name svc_embedding_baseline_run01_20260430_232020 --no-pickle_models --no-write

##12. Large Random Search On Best LL Model - Logistic Regression

In [ ]:
from scipy.stats import loguniform

N_RUNS_LARGE  = 30
RS_SEED_LARGE = 43
np_rng_large  = np.random.default_rng(RS_SEED_LARGE)
lr_ll_large_tags = []

penalties = ["l1", "l2"]

for i in range(N_RUNS_LARGE):
    C       = float(10 ** np_rng_large.uniform(np.log10(1e-4), np.log10(100)))
    penalty = penalties[int(np_rng_large.integers(0, 2))]
    tag     = make_run_tag('lr_embedding_baseline_large', i)
    lr_ll_large_tags.append(tag)

    print(f'\n--- Large LR (LL) run {i:03d} | C={C:.5f}  penalty={penalty} ---')
    run_cmd([
        'python', 'train_lr_embedding_baseline.py',
        '--dataset',            DATASET,
        '--C',                  f'{C:.8f}',
        '--penalty',            penalty,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED_LARGE),
        '--run_name',           tag,
        '--no-pickle_models',
        '--no-write_predictions',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:03d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:03d} done (no metrics found)')

print('\nPhase 1 complete.')
print(lr_ll_large_tags)


--- Large LR (LL) run 000 | C=0.81997  penalty=l1 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.81996549 --penalty l1 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 43 --run_name lr_embedding_baseline_large_run00_20260501_011458 --no-pickle_models --no-write_predictions
  Run 000 done | donor micro_aupr = 0.9601
  Run 000 done | acceptor micro_aupr = 0.6546

--- Large LR (LL) run 001 | C=0.00013  penalty=l1 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.00013188 --penalty l1 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 43 --run_name lr_embedding_baseline_large_run01_20260501_011607 --no-pickle_models --no-write_predictions
  Run 001 done | donor micro_aupr = 0.5301
  Run 001 done | acceptor micro_aupr = 0.0548

--- Large LR (LL) run 002 | C=10.84615  penalty=l1 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 10.84614722 --penalty l1 --cv_mode kfold --n_splits 5 --

## 13. Refined Random Search on best LL Model - Logistic Regression

In [ ]:
#Set these ranges
BEST_PENALTY = "l2"
LOW_REFINED  = 1e-2
HIGH_REFINED = 1e-1

N_RUNS_REFINED  = 15
RS_SEED_REFINED = 44
np_rng_refined  = np.random.default_rng(RS_SEED_REFINED)
lr_ll_refined_tags = []

for i in range(N_RUNS_REFINED):
    C   = float(10 ** np_rng_refined.uniform(np.log10(LOW_REFINED), np.log10(HIGH_REFINED)))
    tag = make_run_tag('lr_ll_refined', i)
    lr_ll_refined_tags.append(tag)

    print(f'\n--- Refined LR (LL) run {i:02d} | C={C:.6f}  penalty={BEST_PENALTY} ---')
    run_cmd([
        'python', 'train_lr_embedding_baseline.py',
        '--dataset',            DATASET,
        '--C',                  f'{C:.8f}',
        '--penalty',            BEST_PENALTY,
        '--cv_mode',            CV_MODE,
        '--n_splits',           str(N_SPLITS),
        '--device',             DEVICE,
        '--min_positive_count', str(MIN_POSITIVE_COUNT),
        '--random_seed',        str(RS_SEED_REFINED),
        '--run_name',           tag,
        '--no-pickle_models',
        '--no-write_predictions',
    ])
    try:
        for task in ['donor', 'acceptor']:
            m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
            with open(m_path) as f:
                m = json.load(f)
            print(f'  Run {i:02d} done | {task} micro_aupr = {m["metrics"]["micro_aupr"]:.4f}')
    except Exception:
        print(f'  Run {i:02d} done (no metrics found)')

print('\nRefined search complete.')
print(lr_ll_refined_tags)


--- Refined LR (LL) run 00 | C=0.013261  penalty=l2 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.01326067 --penalty l2 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 44 --run_name lr_ll_refined_run00_20260501_122848 --no-pickle_models --no-write_predictions
  Run 00 done | donor micro_aupr = 0.9620
  Run 00 done | acceptor micro_aupr = 0.7285

--- Refined LR (LL) run 01 | C=0.018118  penalty=l2 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.01811812 --penalty l2 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 44 --run_name lr_ll_refined_run01_20260501_122954 --no-pickle_models --no-write_predictions
  Run 01 done | donor micro_aupr = 0.9623
  Run 01 done | acceptor micro_aupr = 0.7309

--- Refined LR (LL) run 02 | C=0.025455  penalty=l2 ---

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.02545486 --penalty l2 --cv_mode kfold --n_splits 5 --device auto --min_positive_c

#14. Final LL

In [ ]:
#WINNING HYPERPARAMETERS
BEST_C = 0.02578700  # refined run11
BEST_PENALTY = "l2"


In [ ]:
#FINAL RUN - 5 FOLD CV
FINAL_LR_LL_5FOLD_TAG = make_run_tag('lr_ll_final_5fold', 0)
print(f'Running 5-fold CV | C={BEST_C}  penalty={BEST_PENALTY} | tag={FINAL_LR_LL_5FOLD_TAG}')
run_cmd([
    'python', 'train_lr_embedding_baseline.py',
    '--dataset',            DATASET,
    '--C',                  f'{BEST_C:.8f}',
    '--penalty',            BEST_PENALTY,
    '--cv_mode',            'kfold',
    '--n_splits',           str(N_SPLITS),
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_LR_LL_5FOLD_TAG,
    '--no-pickle_models',
    '--write_predictions',
])
for task in ['donor', 'acceptor']:
    m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / FINAL_LR_LL_5FOLD_TAG / f'bahd_{task}_metrics.json'
    with open(m_path) as f:
        m = json.load(f)['metrics']
    print(f'  5-fold | {task:8s} | micro_aupr={m["micro_aupr"]:.4f}  macro_aupr={m["macro_aupr"]:.4f}  micro_auroc={m["micro_auroc"]:.4f}  macro_auroc={m["macro_auroc"]:.4f}')

Running 5-fold CV | C=0.025787  penalty=l2 | tag=lr_ll_final_5fold_run00_20260509_200740

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.02578700 --penalty l2 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 45 --run_name lr_ll_final_5fold_run00_20260509_200740 --no-pickle_models --write_predictions
  5-fold | donor    | micro_aupr=0.9637  macro_aupr=0.9645  micro_auroc=0.9616  macro_auroc=0.9630
  5-fold | acceptor | micro_aupr=0.7356  macro_aupr=0.6084  micro_auroc=0.9242  macro_auroc=0.9149


In [ ]:
#FINAL RUN - LOOCV
FINAL_LR_LL_LOOCV_TAG = make_run_tag('lr_ll_final_loocv', 0)
print(f'Running LOOCV | C={BEST_C}  penalty={BEST_PENALTY} | tag={FINAL_LR_LL_LOOCV_TAG}')
run_cmd([
    'python', 'train_lr_embedding_baseline.py',
    '--dataset',            DATASET,
    '--C',                  f'{BEST_C:.8f}',
    '--penalty',            BEST_PENALTY,
    '--cv_mode',            'loocv',
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_LR_LL_LOOCV_TAG,
    '--no-pickle_models',
    '--write_predictions',
])
for task in ['donor', 'acceptor']:
    m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / FINAL_LR_LL_LOOCV_TAG / f'bahd_{task}_metrics.json'
    with open(m_path) as f:
        m = json.load(f)['metrics']
    print(f'  LOOCV  | {task:8s} | micro_aupr={m["micro_aupr"]:.4f}  macro_aupr={m["macro_aupr"]:.4f}  micro_auroc={m["micro_auroc"]:.4f}  macro_auroc={m["macro_auroc"]:.4f}')

Running LOOCV | C=0.025787  penalty=l2 | tag=lr_ll_final_loocv_run00_20260509_200849

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.02578700 --penalty l2 --cv_mode loocv --device auto --min_positive_count 2 --random_seed 45 --run_name lr_ll_final_loocv_run00_20260509_200849 --no-pickle_models --write_predictions


KeyboardInterrupt: 

In [ ]:
#FINAL RUN FOLD COMPARISON
print(f'{"":20s} {"Micro AUPR":>10} {"Macro AUPR":>10} {"Micro AUROC":>11} {"Macro AUROC":>11}')
print('-' * 65)
for tag, label in [(FINAL_LR_LL_5FOLD_TAG, '5-fold'), (FINAL_LR_LL_LOOCV_TAG, 'LOOCV')]:
    for task in ['donor', 'acceptor']:
        m_path = ML_DIR / 'results' / 'lr_embedding_baseline' / tag / f'bahd_{task}_metrics.json'
        with open(m_path) as f:
            m = json.load(f)['metrics']
        print(f'{label} {task:8s}          {m["micro_aupr"]:>10.4f} {m["macro_aupr"]:>10.4f} {m["micro_auroc"]:>11.4f} {m["macro_auroc"]:>11.4f}')
    print()

                     Micro AUPR Macro AUPR Micro AUROC Macro AUROC
-----------------------------------------------------------------
5-fold donor                 0.9637     0.9645      0.9616      0.9630
5-fold acceptor              0.7356     0.6084      0.9242      0.9149

LOOCV donor                 0.9686     0.9694      0.9667      0.9674


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CS_5782_Final_Project/code_and_data/results/lr_embedding_baseline/lr_ll_final_loocv_run00_20260509_200849/bahd_acceptor_metrics.json'

In [ ]:
#FULL DATA REFIN & PICKLING
FINAL_LR_LL_FULL_TAG = make_run_tag('lr_ll_final_full', 0)
print(f'Full-data refit | C={BEST_C}  penalty={BEST_PENALTY} | tag={FINAL_LR_LL_FULL_TAG}')
run_cmd([
    'python', 'train_lr_embedding_baseline.py',
    '--dataset',            DATASET,
    '--C',                  f'{BEST_C:.8f}',
    '--penalty',            BEST_PENALTY,
    '--cv_mode',            'kfold',
    '--n_splits',           str(N_SPLITS),
    '--device',             DEVICE,
    '--min_positive_count', str(MIN_POSITIVE_COUNT),
    '--random_seed',        '45',
    '--run_name',           FINAL_LR_LL_FULL_TAG,
    '--pickle_models',
    '--no-write_predictions',
])
print(f'Model saved to: results/lr_embedding_baseline/{FINAL_LR_LL_FULL_TAG}/')

Full-data refit | C=0.025787  penalty=l2 | tag=lr_ll_final_full_run00_20260509_201316

$ python train_lr_embedding_baseline.py --dataset BAHD --C 0.02578700 --penalty l2 --cv_mode kfold --n_splits 5 --device auto --min_positive_count 2 --random_seed 45 --run_name lr_ll_final_full_run00_20260509_201316 --pickle_models --no-write_predictions
Model saved to: results/lr_embedding_baseline/lr_ll_final_full_run00_20260509_201316/
